# Consolidación Usuarios

## 1. Configuración y librerías

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── AJUSTA ESTA RUTA ──────────────────────────────────────────────
BASE_PATH   = 'Sweet_data_Subset'
OUTPUT_PATH = 'sweet_55_usuarios.csv'
N_USUARIOS  = 55
# ─────────────────────────────────────────────────────────────────

QI_MIN          = 0.8    # calidad mínima ECG
ACC_STD_MAX     = 0.10   # percentil 50 de ACC_std_mean
VENTANA_MINUTOS = 5      # minutos antes de cada reporte

print(f'Base path existe: {os.path.exists(BASE_PATH)}')

# Detectar carpetas de usuarios automáticamente
user_folders = sorted([
    f for f in os.listdir(BASE_PATH)
    if os.path.isdir(os.path.join(BASE_PATH, f)) and f.startswith('user')
])
print(f'Usuarios detectados: {len(user_folders)}')
print(f'Primeros 5: {user_folders[:5]}')

usuarios_seleccionados = user_folders[:N_USUARIOS]
print(f'\nUsuarios a procesar ({N_USUARIOS}): {usuarios_seleccionados}')

Base path existe: True
Usuarios detectados: 55
Primeros 5: ['user0091', 'user0093', 'user0096', 'user0098', 'user0099']

Usuarios a procesar (55): ['user0091', 'user0093', 'user0096', 'user0098', 'user0099', 'user0103', 'user0105', 'user0125', 'user0129', 'user0132', 'user0135', 'user0141', 'user0151', 'user0160', 'user0166', 'user0170', 'user0174', 'user0183', 'user0184', 'user0186', 'user0188', 'user0192', 'user0195', 'user0197', 'user0207', 'user0208', 'user0211', 'user0215', 'user0220', 'user0221', 'user0226', 'user0227', 'user0234', 'user0245', 'user0249', 'user1025', 'user1026', 'user1027', 'user1028', 'user1040', 'user1065', 'user1067', 'user1069', 'user1077', 'user1089', 'user1091', 'user1092', 'user1105', 'user1108', 'user1120', 'user1121', 'user1129', 'user1138', 'user1144', 'user1150']


## 2. Funciones auxiliares

In [2]:
def leer_csv_ts(path):
    """Lee CSV donde la primera columna (sin nombre) es el timestamp."""
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = 'timestamp'
    return df.sort_index()


def parsear_lista(valor):
    """Convierte '[ 2.  4.]' → [2, 4]. NaN → []"""
    if pd.isna(valor) or str(valor).strip() == '':
        return []
    try:
        limpio = str(valor).strip().replace('[','').replace(']','')
        return [int(float(x)) for x in limpio.split() if x.strip()]
    except:
        return []


def stats_ventana(serie, prefijo):
    """Estadísticos de una señal en ventana. None si < 3 valores."""
    if len(serie) < 3:
        return None
    return {
        f'{prefijo}_mean'  : serie.mean(),
        f'{prefijo}_std'   : serie.std(),
        f'{prefijo}_median': serie.median(),
        f'{prefijo}_min'   : serie.min(),
        f'{prefijo}_max'   : serie.max(),
    }


def cortar_ventana(df, t_fin, minutos):
    """Devuelve filas de df en [t_fin - minutos, t_fin]."""
    t0 = t_fin - pd.Timedelta(minutes=minutos)
    return df.loc[(df.index >= t0) & (df.index <= t_fin)]


def agrupar_stress(v):
    """1→S1, 2→S2, 3/4/5→S3 (igual que el paper SWEET)"""
    if v == 1: return 'S1'
    if v == 2: return 'S2'
    return 'S3'


def procesar_usuario(user_path, user_id):
    """
    Procesa un usuario completo:
    carga, merge por ventana, filtros y etiquetas.
    Retorna (DataFrame, estado)
    """
    # ── Features ECG + ACC pre-calculadas ─────────────────────
    days = []
    for day in range(5):
        p = os.path.join(user_path, f'{user_id}_Features_Day{day}.csv')
        if os.path.exists(p):
            try:
                days.append(leer_csv_ts(p))
            except:
                pass
    if len(days) == 0:
        return None, 'sin_features'
    features_df = pd.concat(days).sort_index()

    # ── Señales crudas ─────────────────────────────────────────
    try:
        acc_df  = leer_csv_ts(os.path.join(user_path, 'MF_ACC.csv'))
        acc_df['magnitude'] = np.sqrt(
            acc_df['x']**2 + acc_df['y']**2 + acc_df['z']**2
        )
    except:
        acc_df = None

    try:
        gsr_df  = leer_csv_ts(os.path.join(user_path, 'MF_GSR.csv'))
        COL_GSR = gsr_df.columns[0]
    except:
        gsr_df  = None
        COL_GSR = None

    try:
        temp_df  = leer_csv_ts(os.path.join(user_path, 'MF_T.csv'))
        COL_TEMP = temp_df.columns[0]
    except:
        temp_df  = None
        COL_TEMP = None

    # ── Reportes de estrés ─────────────────────────────────────
    stress_path = os.path.join(user_path, 'current_stress.csv')
    if not os.path.exists(stress_path):
        return None, 'sin_stress'
    try:
        stress_df = leer_csv_ts(stress_path)
        stress_df = stress_df.dropna(subset=['MAXIMUM_STRESS'])
        stress_df['ACTIVITY_list']    = stress_df['ACTIVITY'].apply(parsear_lista)
        stress_df['CONSUMPTION_list'] = stress_df['CONSUMPTION'].apply(parsear_lista)
    except:
        return None, 'error_stress'

    if len(stress_df) == 0:
        return None, 'sin_reportes'

    # ── Merge por ventana de 5 minutos ─────────────────────────
    filas = []
    for t_rep, reporte in stress_df.iterrows():
        v_ecg = cortar_ventana(features_df, t_rep, VENTANA_MINUTOS)
        if len(v_ecg) == 0:
            continue

        fila = v_ecg.mean(numeric_only=True).to_dict()

        if gsr_df is not None:
            v_gsr = cortar_ventana(gsr_df, t_rep, VENTANA_MINUTOS)
            feats = stats_ventana(v_gsr[COL_GSR], 'GSR')
            if feats: fila.update(feats)

        if temp_df is not None:
            v_tmp = cortar_ventana(temp_df, t_rep, VENTANA_MINUTOS)
            feats = stats_ventana(v_tmp[COL_TEMP], 'TEMP')
            if feats: fila.update(feats)

        if acc_df is not None:
            v_acc = cortar_ventana(acc_df, t_rep, VENTANA_MINUTOS)
            feats = stats_ventana(v_acc['magnitude'], 'ACC_mag')
            if feats:
                fila['ACC_mag_std']  = feats['ACC_mag_std']
                fila['ACC_mag_mean'] = feats['ACC_mag_mean']

        fila['user_id']          = user_id
        fila['timestamp']        = t_rep
        fila['MAXIMUM_STRESS']   = reporte['MAXIMUM_STRESS']
        fila['PLEASURE']         = reporte.get('PLEASURE', np.nan)
        fila['AROUSAL']          = reporte.get('AROUSAL', np.nan)
        fila['DOMINANCE']        = reporte.get('DOMINANCE_CONTROL', np.nan)
        fila['ACTIVITY_list']    = str(reporte['ACTIVITY_list'])
        fila['CONSUMPTION_list'] = str(reporte['CONSUMPTION_list'])
        filas.append(fila)

    if len(filas) == 0:
        return None, 'sin_merge'

    df_user = pd.DataFrame(filas)

    # ── Filtro 1: calidad ECG ──────────────────────────────────
    if 'ECG_QI_mean' in df_user.columns:
        df_user = df_user[df_user['ECG_QI_mean'] >= QI_MIN]

    if len(df_user) == 0:
        return None, 'sin_datos_tras_qi'

    # ── Filtro 2: actividad física (umbral fijo p50) ───────────
    # 0.10 = percentil 50 de ACC_std_mean para user1150
    # Elimina momentos de actividad media-alta que confunden
    # la señal fisiológica de estrés
    df_user['ACC_std_mean'] = (
        df_user['std_x'] + df_user['std_y'] + df_user['std_z']
    ) / 3
    df_user = df_user[df_user['ACC_std_mean'] <= ACC_STD_MAX]

    if len(df_user) == 0:
        return None, 'sin_datos_tras_acc'

    # ── Etiquetas S1 / S2 / S3 ────────────────────────────────
    df_user['stress_label'] = df_user['MAXIMUM_STRESS'].apply(agrupar_stress)

    return df_user, 'ok'


print('✓ Funciones cargadas')

✓ Funciones cargadas


## 3. Loop sobre 30 usuarios

In [3]:
todos   = []
errores = {}

for i, user_id in enumerate(usuarios_seleccionados):
    user_path = os.path.join(BASE_PATH, user_id)
    df_user, estado = procesar_usuario(user_path, user_id)

    if estado == 'ok':
        todos.append(df_user)
        s1 = (df_user['stress_label'] == 'S1').sum()
        s2 = (df_user['stress_label'] == 'S2').sum()
        s3 = (df_user['stress_label'] == 'S3').sum()
        print(f'  [{i+1:>2}/{N_USUARIOS}] ✓ {user_id:12} '
              f'→ {len(df_user):>3} filas  '
              f'(S1:{s1} S2:{s2} S3:{s3})')
    else:
        errores[estado] = errores.get(estado, 0) + 1
        print(f'  [{i+1:>2}/{N_USUARIOS}] ✗ {user_id:12} → {estado}')

print(f'\nUsuarios OK:    {len(todos)} / {N_USUARIOS}')
if errores:
    print(f'Errores:        {errores}')

  [ 1/55] ✓ user0091     →   5 filas  (S1:5 S2:0 S3:0)
  [ 2/55] ✓ user0093     →  16 filas  (S1:15 S2:0 S3:1)
  [ 3/55] ✓ user0096     →  18 filas  (S1:11 S2:6 S3:1)
  [ 4/55] ✓ user0098     →   5 filas  (S1:1 S2:4 S3:0)
  [ 5/55] ✓ user0099     →  25 filas  (S1:8 S2:14 S3:3)
  [ 6/55] ✓ user0103     →  18 filas  (S1:8 S2:4 S3:6)
  [ 7/55] ✓ user0105     →  10 filas  (S1:6 S2:3 S3:1)
  [ 8/55] ✓ user0125     →  17 filas  (S1:16 S2:1 S3:0)
  [ 9/55] ✓ user0129     →  17 filas  (S1:7 S2:5 S3:5)
  [10/55] ✓ user0132     →  10 filas  (S1:8 S2:1 S3:1)
  [11/55] ✓ user0135     →  15 filas  (S1:10 S2:2 S3:3)
  [12/55] ✓ user0141     →  12 filas  (S1:6 S2:6 S3:0)
  [13/55] ✓ user0151     →  13 filas  (S1:5 S2:8 S3:0)
  [14/55] ✓ user0160     →   8 filas  (S1:5 S2:3 S3:0)
  [15/55] ✓ user0166     →  10 filas  (S1:8 S2:0 S3:2)
  [16/55] ✓ user0170     →   3 filas  (S1:3 S2:0 S3:0)
  [17/55] ✓ user0174     →  11 filas  (S1:7 S2:3 S3:1)
  [18/55] ✓ user0183     →  20 filas  (S1:7 S2:9 S3:4)
  [19

## 4. Tabla consolidada — resumen

In [4]:
df_total = pd.concat(todos, ignore_index=True)

print(f'Filas totales:    {len(df_total):,}')
print(f'Columnas totales: {len(df_total.columns)}')
print(f'Usuarios:         {df_total["user_id"].nunique()}')

Filas totales:    770
Columnas totales: 37
Usuarios:         55


In [5]:
# Distribución global de clases
print('Distribución global stress_label:')
nombres = {'S1':'Sin estrés','S2':'Leve','S3':'Alto'}
for label in ['S1','S2','S3']:
    n   = (df_total['stress_label'] == label).sum()
    pct = n / len(df_total) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label} {nombres[label]:12} {n:>4} ({pct:5.1f}%) {bar}')

Distribución global stress_label:
  S1 Sin estrés    443 ( 57.5%) ████████████████████████████
  S2 Leve          220 ( 28.6%) ██████████████
  S3 Alto          107 ( 13.9%) ██████


In [6]:
# Filas por usuario
filas_por_user = df_total.groupby('user_id').size().sort_values()
print('Filas por usuario:')
print(f'  Mínimo:   {filas_por_user.min()}')
print(f'  Máximo:   {filas_por_user.max()}')
print(f'  Promedio: {filas_por_user.mean():.1f}')
print(f'\nUsuarios con < 5 filas: {(filas_por_user < 5).sum()}')
print()
print(filas_por_user.to_string())

Filas por usuario:
  Mínimo:   3
  Máximo:   25
  Promedio: 14.0

Usuarios con < 5 filas: 2

user_id
user0170     3
user0220     4
user0091     5
user0098     5
user0184     5
user1069     7
user0221     8
user1077     8
user0160     8
user0207     9
user0249     9
user1091     9
user0132    10
user0166    10
user1120    10
user0105    10
user0195    11
user0174    11
user0197    11
user1129    12
user0141    12
user0245    13
user1121    13
user1040    13
user0151    13
user0226    13
user1108    13
user1089    13
user0208    14
user1025    14
user0135    15
user0211    15
user1138    15
user1067    16
user1028    16
user0215    16
user0093    16
user0129    17
user0125    17
user1027    18
user0096    18
user1065    18
user0103    18
user0188    18
user1105    19
user1026    20
user0183    20
user0186    21
user1144    21
user1150    21
user1092    23
user0192    23
user0227    23
user0099    25
user0234    25


In [7]:
# NaN por columna
nulos = df_total.isnull().sum()
nulos = nulos[nulos > 0]
if len(nulos) > 0:
    print('Columnas con NaN:')
    print(nulos)
else:
    print('✓ Sin NaN en la tabla consolidada')

Columnas con NaN:
('ECG_mean_heart_rate', 'raw0')    770
PLEASURE                             7
AROUSAL                              5
DOMINANCE                            2
dtype: int64


In [8]:
# Features disponibles por tipo de señal
ecg_cols  = [c for c in df_total.columns if 'ECG'  in str(c)]
gsr_cols  = [c for c in df_total.columns if 'GSR'  in str(c)]
temp_cols = [c for c in df_total.columns if 'TEMP' in str(c)]
acc_cols  = [c for c in df_total.columns if 'ACC'  in str(c)
                                         or str(c) in ['std_x','std_y','std_z',
                                                        'mean_x','mean_y','mean_z']]
print(f'ECG  ({len(ecg_cols)}): {ecg_cols}')
print(f'GSR  ({len(gsr_cols)}): {gsr_cols}')
print(f'TEMP ({len(temp_cols)}): {temp_cols}')
print(f'ACC  ({len(acc_cols)}): {acc_cols}')

ECG  (8): ['ECG_mean_heart_rate', 'ECG_sdnn', 'ECG_rmssd', 'ECG_LF', 'ECG_HF', 'ECG_LFHF', "('ECG_mean_heart_rate', 'raw0')", 'ECG_QI_mean']
GSR  (5): ['GSR_mean', 'GSR_std', 'GSR_median', 'GSR_min', 'GSR_max']
TEMP (5): ['TEMP_mean', 'TEMP_std', 'TEMP_median', 'TEMP_min', 'TEMP_max']
ACC  (9): ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z', 'ACC_mag_std', 'ACC_mag_mean', 'ACC_std_mean']


In [9]:
# Preview de la tabla consolidada
cols_preview = ['user_id','timestamp','ECG_mean_heart_rate',
                'ECG_rmssd','GSR_mean','TEMP_median',
                'MAXIMUM_STRESS','stress_label']
cols_ok = [c for c in cols_preview if c in df_total.columns]
df_total[cols_ok].head(10)

,user_id,timestamp,ECG_mean_heart_rate,ECG_rmssd,GSR_mean,TEMP_median,MAXIMUM_STRESS,stress_label
0,user0091,2015-12-04 11:05:22,69.147348,872.896392,1.159803,31.0,1.0,S1
1,user0091,2015-12-04 14:37:43,80.453756,747.529235,1.351838,32.0,1.0,S1
2,user0091,2015-12-04 20:57:59,84.092332,716.085356,1.065842,31.0,1.0,S1
3,user0091,2015-12-06 20:51:46,94.257167,637.785272,0.386528,31.0,1.0,S1
4,user0091,2015-12-07 14:06:26,86.105870,699.409036,0.361877,34.0,1.0,S1
5,user0093,2015-12-03 11:09:32,78.673802,765.130595,6.499247,30.0,3.0,S3
6,user0093,2015-12-03 15:13:20,64.375379,932.907528,2.669158,32.0,1.0,S1
7,user0093,2015-12-03 16:27:44,60.176207,997.830283,0.325936,30.0,1.0,S1
8,user0093,2015-12-04 18:49:20,56.290814,1067.023153,0.452265,29.0,1.0,S1
9,user0093,2015-12-04 20:28:01,73.796989,814.884095,-5.975495,32.0,1.0,S1


## 5. Guardado

In [10]:
df_total.to_csv(OUTPUT_PATH, index=False)
print(f'✓ Guardado en: {OUTPUT_PATH}')
print(f'  Filas:    {len(df_total):,}')
print(f'  Columnas: {len(df_total.columns)}')

✓ Guardado en: sweet_55_usuarios.csv
  Filas:    770
  Columnas: 37

Siguiente paso → 03_EDA.ipynb
